# Assistente de Investimentos com AI Agent: VALE3, PETR4, BBAS3, ITUB4

> **Aviso:** projeto experimental e educacional. As recomendações **não constituem aconselhamento financeiro** e **não garantem resultados**. Resultados históricos não indicam desempenho futuro.

**Como o agente decide:** coleta preços (`yfinance`) e notícias (`feedparser`), calcula indicadores técnicos, pontua o sentimento por léxico e aplica regras auditáveis. O registro guarda apenas evidências, regras acionadas e limitações (sem raciocínio interno do modelo). Nenhuma chave paga é necessária.

## 1. Preparação do ambiente

### 1.1. Configuração das chaves de API

O fluxo principal **não usa chaves**. As variáveis abaixo são opcionais e reservadas para integrações futuras com LLM (não implementadas neste notebook):

- `OPENAI_API_KEY`: para o modelo OpenAI
- `GOOGLE_API_KEY`: para o Google ADK

Defina-as no ambiente ou nos *Secrets* do Colab; nunca escreva chaves no notebook.

In [ ]:
import os

def optional_secret(name):
    """Lê a chave do ambiente ou dos Secrets do Colab; retorna None se ausente."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

for key in ("OPENAI_API_KEY", "GOOGLE_API_KEY"):
    print(key, "definida" if optional_secret(key) else "não definida (opcional)")

### 1.2. Instalação das dependências

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Colab já traz pandas/numpy/matplotlib; instala apenas o que falta.
    %pip install -q yfinance feedparser
else:
    print("Execução local: instale com `pip install -r requirements` (ver README.md).")

# Integrações opcionais com LLM (não usadas no fluxo principal):
# %pip install openai google-api-python-client

### 1.3. Código do agente (pacote `assistente/`)

- **Local:** a pasta `assistente/` já está ao lado do notebook.
- **Colab:** compacte a pasta em `assistente.zip` e envie quando solicitado abaixo.

In [ ]:
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("assistente") is None:
    if not IN_COLAB:
        raise ModuleNotFoundError("Pasta 'assistente/' não encontrada; abra o notebook na raiz do projeto.")
    from google.colab import files
    print("Envie o arquivo assistente.zip (contendo a pasta assistente/):")
    for name in files.upload():
        zipfile.ZipFile(name).extractall(".")
    sys.path.insert(0, str(pathlib.Path(".").resolve()))
    importlib.invalidate_caches()
    if importlib.util.find_spec("assistente") is None:
        raise ModuleNotFoundError("O zip enviado não contém a pasta 'assistente/' na raiz.")
print("Pacote 'assistente' disponível.")

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt

from assistente import config
from assistente.agent import TOOLS, InvestmentAgent
from assistente.chat import answer
from assistente.data import get_price_data
from assistente.evaluation import evaluate_ticker
from assistente.indicators import calculate_indicators
from assistente.news import search_news
from assistente.plots import plot_backtest, plot_indicators

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
pd.set_option("display.width", 140)
print(config.DISCLAIMER)

## 2. Ferramentas do agente

| Ferramenta | Função |
| --- | --- |
| `search_news(ticker)` | notícias por RSS relacionadas à ação, com sentimento |
| `get_price_data(ticker, period)` | OHLCV diário via `yfinance` |
| `calculate_indicators(data)` | RSI, MACD, SMA/EMA, Bollinger e volume |
| `generate_recommendation(analysis)` | COMPRAR/VENDER/AGUARDAR com evidências |

In [ ]:
for t in TOOLS.values():
    print(f"{t.name}: {t.description}")

### 2.1. Preços e indicadores (exemplo: PETR4)

In [ ]:
prices = get_price_data("PETR4", "2y")
if prices.ok:
    ind = calculate_indicators(prices.data)
    display(ind[["Close", "Volume", "RSI", "MACD", "MACD_signal", "SMA_short", "SMA_long",
                 "BB_upper", "BB_lower"]].tail())
else:
    print("Preços indisponíveis:", prices.error)

### 2.2. Notícias e sentimento (exemplo: PETR4)

O sentimento usa um **léxico em português**: gratuito e offline, mas não entende ironia nem contexto. Só títulos e resumos dos feeds são usados; sem notícias relacionadas, o score fica indefinido.

In [ ]:
news = search_news("PETR4")
print("Fontes OK:", news.sources_ok)
print("Fontes com falha:", news.sources_failed)
print(news.summary)
pd.DataFrame(news.items)[["source", "title", "label", "score"]] if news.items else None

## 3. Agente: análise das quatro ações

In [ ]:
agent = InvestmentAgent(period="2y")
agent.run_all()
summary = pd.DataFrame([
    {k: r[k] for k in ("ticker", "date", "close", "rsi", "macd_signal", "news_sentiment", "recommendation", "score")}
    for r in agent.records.values()
])
display(summary)
print("Registro salvo em:", agent.save("recomendacoes.json"))

In [ ]:
for r in agent.records.values():
    print(f"=== {r['ticker']} ({r['date']}): {r['recommendation']} ===")
    for s in r["trace"]:
        print(f"  [{s['passo']}] {s['ferramenta']}({s['entrada']}) -> {s['resultado']}")
    print("  Justificativa:", r["justification"])
    print("  Notícias:", r["news_summary"])
    print("  Limitações:", " | ".join(r["limitations"]))
    print()

## 4. Visualizações

In [ ]:
for t in config.STOCKS:
    res = get_price_data(t, "2y")
    if res.ok:
        plot_indicators(calculate_indicators(res.data), t)
        plt.show()
    else:
        print(f"{t}: gráfico indisponível ({res.error})")

## 5. Interface conversacional

As respostas usam apenas os registros gerados acima.

In [ ]:
for q in ["Por que você recomendou isso para PETR4?",
          "Quais notícias foram consideradas para VALE3?",
          "Como estão RSI e MACD de ITUB4?",
          "Quais as limitações da recomendação de BBAS3?"]:
    print("Pergunta:", q)
    print(answer(q, agent.records), "\n")

## 6. Avaliação histórica e backtest

**Definições** (também em `assistente/evaluation.py`):

- Cada decisão em D usa só indicadores calculados com dados **até o fechamento de D** (sem vazamento); testes automatizados verificam isso.
- Somente sinais técnicos são avaliados: o RSS não fornece histórico de notícias confiável.
- **Acerto de tendência** (horizonte de 5 pregões): COMPRAR acerta se o retorno de D a D+5 for > 0; VENDER, se < 0; AGUARDAR, se |retorno| < 2%.
- Decisões sem D+5 disponível não são avaliadas (são contadas à parte).
- **Backtest long-only:** COMPRAR posiciona, VENDER sai, AGUARDAR mantém a posição; a decisão de D rende a partir de D+1; custo de 0,1% por operação; comparado ao Buy-and-Hold no mesmo intervalo.
- **Limitações:** sem impostos, slippage, dividendos separados (preços ajustados) nem otimização de parâmetros; amostra de ~2 anos e uma única janela. Desempenho passado não garante desempenho futuro.

In [ ]:
eval_results = {}
rows = []
for t in config.STOCKS:
    res = get_price_data(t, "2y")
    if not res.ok:
        print(f"{t}: avaliação ignorada ({res.error})")
        continue
    out = evaluate_ticker(res.data)
    if "erro" in out:
        print(f"{t}: {out['erro']}")
        continue
    eval_results[t] = out
    m, b = out["metrics"], out["backtest"]
    rows.append({"ticker": t, "decisões avaliadas": m["decisoes_avaliadas"],
                 "acurácia geral": m["acuracia_geral"],
                 "acurácia direcional (sem AGUARDAR)": m["acuracia_direcional_sem_aguardar"],
                 "retorno agente": b["retorno_estrategia"], "retorno buy&hold": b["retorno_buy_and_hold"],
                 "operações": b["operacoes"]})
results_df = pd.DataFrame(rows)
if not results_df.empty:
    display(results_df.style.format({c: "{:.1%}" for c in results_df.columns[2:6]}))
else:
    print("Sem dados suficientes para avaliação.")

In [ ]:
for t, out in eval_results.items():
    print(t, {k: v for k, v in out["metrics"]["por_recomendacao"].items()})
    plot_backtest(out["curve"], t)
    plt.show()

## 7. Premissas e limitações

- Tickers B3 via Yahoo Finance (`.SA`), preços ajustados; dados podem ter atraso ou falhas, tratadas sem interromper o fluxo.
- Feeds RSS públicos podem mudar ou ficar indisponíveis; sem notícias, o sentimento não entra na decisão.
- O sentimento por léxico é simples e pode errar; notícias só são relacionadas por nome/ticker.
- Regras e limiares são heurísticos (`assistente/config.py`), não otimizados.
- Dados fundamentalistas, alertas e multiagente não foram implementados.
- Este material é experimental e não é recomendação de investimento.